<h1 align="center">🏥 Insurance Claims Intelligence Workshop</h1>
<h3 align="center">Notebook 00 — Environment Setup</h3>

<div align="center">
<table style="border:none;background:transparent;margin:auto"><tr style="border:none">
<td style="border:none;padding:2px"><img src="https://img.shields.io/badge/python-3.9%2B-blue?logo=python&logoColor=white" /></td>
<td style="border:none;padding:2px"><img src="https://img.shields.io/badge/jupyter-notebook-orange?logo=jupyter&logoColor=white" /></td>
<td style="border:none;padding:2px"><img src="https://img.shields.io/badge/Microsoft%20Fabric-ready-0078D4?logo=microsoft&logoColor=white" /></td>
</tr></table>
</div>

---

> **Purpose:** Run each cell in order to provision all Fabric resources needed for the workshop.  
> Each step is idempotent — safe to re-run if something fails partway through.

## 📋 Prerequisites

#### 🏗️ Workspace & Access

<table style="text-align:left; width:100%">
<thead><tr><th></th><th>Requirement</th><th>Detail</th></tr></thead>
<tbody>
<tr><td>🏢</td><td><strong>Fabric-enabled workspace</strong></td><td>Use this workspace for all resources in the tutorial</td></tr>
<tr><td>👤</td><td><strong>Workspace role: Contributor or higher</strong></td><td>The user running this notebook must have at least Contributor access; Admin role recommended</td></tr>
<tr><td>⚡</td><td><strong>F16 capacity (minimum)</strong></td><td>Use at least F16 to avoid throttling and ensure optimal performance</td></tr>
</tbody>
</table>

#### 🔧 Tenant Settings *(enabled by a Fabric Administrator)*

<table style="text-align:left; width:100%">
<thead><tr><th></th><th>Setting</th><th>Location</th></tr></thead>
<tbody>
<tr><td>1️⃣</td><td>Enable <strong>Ontology item</strong> <em>(preview)</em></td><td>Admin Portal → Tenant Settings</td></tr>
<tr><td>2️⃣</td><td>User can create <strong>Graph</strong> <em>(preview)</em></td><td>Admin Portal → Tenant Settings</td></tr>
<tr><td>3️⃣</td><td>Users can create and share <strong>Data agent</strong> item types <em>(preview)</em></td><td>Admin Portal → Tenant Settings</td></tr>
<tr><td>4️⃣</td><td>Users can use <strong>Copilot</strong> and features powered by Azure OpenAI</td><td>Admin Portal → Tenant Settings</td></tr>
<tr><td>5️⃣</td><td>Data sent to Azure OpenAI can be <strong>processed outside</strong> your capacity's region</td><td>Admin Portal → Tenant Settings</td></tr>
<tr><td>6️⃣</td><td>Data sent to Azure OpenAI can be <strong>stored outside</strong> your capacity's region</td><td>Admin Portal → Tenant Settings</td></tr>
</tbody>
</table>

> ⚠️ Settings 4–6 involve cross-region data processing by Azure OpenAI. Review your organization's data residency and compliance policies before enabling them.


---
## ⚙️ Setup — Configure & Download Assets

Run the two cells below to configure and download all workshop assets:
1. **create_ontology** — set to True if you want to create ontology at set up. Set to False to skip automated ontology creation (create manually via Option 2 in README)
2. **Workspace ID and Name are optional** — if left blank, the notebook will automatically detect them from your current Fabric session. Only fill these in if you want to target a different workspace.

Note: This notebook will download all workshop scripts, reference data, and the `InsuranceSM.SemanticModel` files from GitHub into `./builtin/`

> These downloads only happen once — every subsequent step just uses local files.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────────────
create_ontology = True  # Set to False to skip automated ontology creation (create manually via Option 2 in README)
# ─────────────────────────────────────────────────────────────────────────────────────
# ── Optional Configurations ──────────────────────────────────────────────────────────
# ── These are optional because the code will attempt to infer them if not set ────────
WORKSPACE_ID   = ""    # 👈 Replace with your Fabric workspace GUID
WORKSPACE_NAME = ""  # 👈 Replace with your Fabric workspace name

StatementMeta(, 8d9b95a4-0f6b-43db-acea-cde4d51747fe, 3, Finished, Available, Finished, False)

In [ ]:
if WORKSPACE_ID == "" or WORKSPACE_NAME == "":
    import sempy.fabric as fabric
    import re

    # Get the workspace ID
    WORKSPACE_ID = fabric.get_notebook_workspace_id()

    # Get the workspace name
    WORKSPACE_NAME = fabric.resolve_workspace_name()

    # Validate: only letters, digits, spaces, hyphens, and underscores allowed
    if not re.fullmatch(r"[A-Za-z0-9 _\-]+", WORKSPACE_NAME):
        raise ValueError(
            f"Workspace name '{WORKSPACE_NAME}' contains special characters. "
            "Only letters, digits, spaces, hyphens, and underscores are allowed."
        )

    print(f"Workspace ID: {WORKSPACE_ID}")
    print(f"Workspace Name: {WORKSPACE_NAME}")

StatementMeta(, 8d9b95a4-0f6b-43db-acea-cde4d51747fe, 4, Finished, Available, Finished, False)

Workspace ID: 59bc29de-d7f5-447a-b567-0cb716f5dd8d
Workspace Name: FSI - Fabric IQ


In [7]:
import urllib.request, json
from pathlib import Path

BRANCH  = "main"
REPO = "mohit434demo/FSI-FabricIQ"
RAW     = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"
API     = f"https://api.github.com/repos/{REPO}/git/trees/{BRANCH}?recursive=1"

# Folders to mirror into ./builtin/  →  (repo_prefix, local_subdir)
FOLDERS = [
    ("scripts", ""),
    ("semantic_model_project/InsuranceSM.SemanticModel", "InsuranceSM.SemanticModel"),
    ("reference_data", "reference_data"),
]

# Individual files to copy into ./builtin/  →  (repo_path, local_relative_path)
FILES = [
    ("semantic_model_project/InsuranceSM.pbip", "InsuranceSM.pbip"),
    ("semantic_model_project/InsuranceSM.Report/definition.pbir", "InsuranceSM.Report/definition.pbir"),
    ("semantic_model_project/InsuranceSM.Report/.platform", "InsuranceSM.Report/.platform"),
    ("semantic_model_project/InsuranceSM.SemanticModel/.platform", "InsuranceSM.SemanticModel/.platform"),
    ("semantic_model_project/InsuranceSM.SemanticModel/.pbi/editorSettings.json", "InsuranceSM.SemanticModel/.pbi/editorSettings.json"),
]

BUILTIN = Path("./builtin")
BUILTIN.mkdir(exist_ok=True)

def _download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "FabricNotebook/1.0"})
    with urllib.request.urlopen(req) as r:
        dest.write_bytes(r.read())

# Fetch full tree once
print(f"Fetching file tree for branch '{BRANCH}'...")
req = urllib.request.Request(API, headers={"User-Agent": "FabricNotebook/1.0"})
with urllib.request.urlopen(req) as r:
    tree = json.loads(r.read())["tree"]
print(f"  {len(tree)} total entries in repo.\n")

repo_paths = {e["path"] for e in tree if e["type"] == "blob"}

for repo_prefix, local_sub in FOLDERS:
    blobs = [e for e in tree if e["type"] == "blob" and e["path"].startswith(repo_prefix + "/")]
    print(f"📥 Downloading {len(blobs)} file(s) from '{repo_prefix}/'...")
    dest_root = BUILTIN / local_sub if local_sub else BUILTIN
    for entry in blobs:
        rel = entry["path"][len(repo_prefix) + 1:]   # strip the folder prefix
        dest = dest_root / rel
        url = f"{RAW}/{entry['path']}"
        _download(url, dest)
        print(f"   ✅ {entry['path']}")
    print()

print(f"📥 Downloading {len(FILES)} individual file(s)...")
for repo_path, local_rel in FILES:
    if repo_path not in repo_paths:
        raise FileNotFoundError(f"Repo file not found: {repo_path}")
    dest = BUILTIN / local_rel
    url = f"{RAW}/{repo_path}"
    _download(url, dest)
    print(f"   ✅ {repo_path} -> {local_rel}")
print()

print(f"✅ All assets downloaded → {BUILTIN.resolve()}")

StatementMeta(, 1cc01447-f545-459c-adca-034546de6c08, 9, Finished, Available, Finished, False)

Fetching file tree for branch 'main'...
  52 total entries in repo.

📥 Downloading 5 file(s) from 'scripts/'...
   ✅ scripts/create_insurance_eventhouse.py
   ✅ scripts/create_insurance_sm.py
   ✅ scripts/eventhouse_setup.kql
   ✅ scripts/generate_reference_data.py
   ✅ scripts/validate_data.py

📥 Downloading 16 file(s) from 'semantic_model_project/InsuranceSM.SemanticModel/'...
   ✅ semantic_model_project/InsuranceSM.SemanticModel/.pbi/editorSettings.json
   ✅ semantic_model_project/InsuranceSM.SemanticModel/.platform
   ✅ semantic_model_project/InsuranceSM.SemanticModel/definition.pbism
   ✅ semantic_model_project/InsuranceSM.SemanticModel/definition/database.tmdl
   ✅ semantic_model_project/InsuranceSM.SemanticModel/definition/expressions.tmdl
   ✅ semantic_model_project/InsuranceSM.SemanticModel/definition/model.tmdl
   ✅ semantic_model_project/InsuranceSM.SemanticModel/definition/relationships.tmdl
   ✅ semantic_model_project/InsuranceSM.SemanticModel/definition/tables/adjusters.t

---
## 🏗️ Step 1 — Create Lakehouse

Creates the `lh_insurance` Lakehouse in your workspace.
If it already exists the cell is a no-op — safe to re-run.

In [ ]:
import sempy.fabric as fabric

LAKEHOUSE_NAME = "lh_insurance"          # Lakehouse to create (or reuse if it exists)

client = fabric.FabricRestClient()

# List all lakehouses in the workspace and find ours by name
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/lakehouses")
resp.raise_for_status()
lakehouses = resp.json().get("value", [])
existing = next((lh for lh in lakehouses if lh["displayName"] == LAKEHOUSE_NAME), None)

if existing:
    LAKEHOUSE_ID = existing["id"]
    print(f"ℹ️  Lakehouse '{LAKEHOUSE_NAME}' already exists — skipping creation.")
    print(f"   LAKEHOUSE_ID = {LAKEHOUSE_ID}")
else:
    resp = client.post(
        f"v1/workspaces/{WORKSPACE_ID}/lakehouses",
        json={"displayName": LAKEHOUSE_NAME},
    )
    resp.raise_for_status()
    LAKEHOUSE_ID = resp.json()["id"]
    print(f"✅ Lakehouse created: {LAKEHOUSE_NAME}")
    print(f"   LAKEHOUSE_ID = {LAKEHOUSE_ID}")


StatementMeta(, 11e984c2-05b3-4906-909e-a59cebedc132, 7, Finished, Available, Finished, False)

✅ Lakehouse created: lh_insurance
   LAKEHOUSE_ID = b8583e01-8326-4d29-9720-51ea15cffdee


---
## 📄 Step 2 — Upload Reference Data

Reference data files (already in `./builtin/`) contain 8 synthetic
JSONL files. Upload them directly to the Lakehouse via OneLake:
`Files/reference_data/`

Files are downloaded to `./builtin/` first, then uploaded using `mssparkutils.fs.put()`
— no default lakehouse mount required.

In [18]:
import os
from pathlib import Path

LOCAL_TMP      = Path("./builtin/reference_data")
ONELAKE_DEST   = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{LAKEHOUSE_ID}/Files/reference_data"
)

jsonl_files = sorted(LOCAL_TMP.glob("*.jsonl"))
if not jsonl_files:
    raise RuntimeError("No JSONL files were written. Check the output above for errors.")

print(f"\nUploading {len(jsonl_files)} files to Lakehouse...")
for f in jsonl_files:
    dest = f"{ONELAKE_DEST}/{f.name}"
    mssparkutils.fs.put(dest, f.read_text(encoding="utf-8"), True)
    print(f"   ✅ {f.name:<35} {f.stat().st_size / 1024:>7.1f} KB")

print(f"\n🎉 {len(jsonl_files)} files written to the Lakehouse.")
print(f"   Lakehouse : {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})")
print( "   Next      : Step 3 will create Delta tables from these files.")


StatementMeta(, 1cc01447-f545-459c-adca-034546de6c08, 20, Finished, Available, Finished, False)


Uploading 8 files to Lakehouse...
   ✅ adjusters.jsonl                        11.9 KB
   ✅ asset_inspections.jsonl                40.5 KB
   ✅ claim_events.jsonl                     35.0 KB
   ✅ claims.jsonl                           20.2 KB
   ✅ insured_assets.jsonl                   18.7 KB
   ✅ offices.jsonl                           5.4 KB
   ✅ policies.jsonl                         22.5 KB
   ✅ policyholders.jsonl                     7.0 KB

🎉 8 files written to the Lakehouse.
   Lakehouse : lh_insurance (b8583e01-8326-4d29-9720-51ea15cffdee)
   Next      : Step 3 will create Delta tables from these files.


In [ ]:
# NOTE:
# If you attach the Lakehouse after the notebook has already started,
# Fabric may create a new Spark session and clear Python variables.
# If that happens:
# 1. Attach the target Lakehouse
# 2. Rerun the setup/config cell
# 3. Then run Step 3

In [ ]:
# # Precheck: rerun setup variables if a new Spark session started
# required_vars = ["WORKSPACE_ID", "WORKSPACE_NAME", "LAKEHOUSE_ID", "LAKEHOUSE_NAME"]
# missing = [v for v in required_vars if v not in globals()]
#
# if missing:
#     raise ValueError(
#         f"Missing required variables: {missing}. "
#         "If attaching the Lakehouse started a new Spark session, rerun the setup/config cell first."
#     )
#
# try:
#     spark.sql(f"SHOW TABLES IN {LAKEHOUSE_NAME}").show(1, truncate=False)
# except Exception as e:
#     raise ValueError(
#         f"Lakehouse '{LAKEHOUSE_NAME}' is not attached or not accessible in this Spark session. "
#         "Attach it in the notebook UI, then rerun the setup/config cell."
#     ) from e

In [ ]:
required_vars = ["WORKSPACE_ID", "WORKSPACE_NAME", "LAKEHOUSE_ID", "LAKEHOUSE_NAME"]
missing = [v for v in required_vars if v not in globals()]

if missing:
    raise ValueError(
        f"Missing required variables: {missing}. "
        "Rerun the setup/config cell at the top of the notebook, or use the restore-context cell below."
    )

---
## 📊 Step 3 — Create Delta Tables

Runs `01_load_reference_data.ipynb` as a child notebook, passing the Lakehouse name and data path as parameters. That notebook reads each JSONL file and creates a corresponding Delta table in the Lakehouse.

> This step may take 2–5 minutes to complete.

In [26]:
notebookutils.notebook.run(
    "01_load_reference_data",
    timeout_seconds=600,
    arguments={
        "LAKEHOUSE_NAME":      LAKEHOUSE_NAME,
        "WORKSPACE_NAME": WORKSPACE_NAME,
        "REFERENCE_DATA_PATH": "Files/reference_data",
        "LAKEHOUSE_ID": LAKEHOUSE_ID,
        "WORKSPACE_ID": WORKSPACE_ID
    },
)
print("✅ Delta tables created successfully.")

StatementMeta(, 1cc01447-f545-459c-adca-034546de6c08, 28, Finished, Available, Finished, False)

✅ Delta tables created successfully.


In [3]:
WORKSPACE_ID = "59bc29de-d7f5-447a-b567-0cb716f5dd8d"
WORKSPACE_NAME = "FSI - Fabric IQ"
LAKEHOUSE_NAME = "lh_insurance"
LAKEHOUSE_ID = "b8583e01-8326-4d29-9720-51ea15cffdee"

StatementMeta(, 07cefa52-e009-41d4-ae35-3b50dc96b221, 5, Finished, Available, Finished, False)

In [27]:
spark.table("adjusters").printSchema()

StatementMeta(, 1cc01447-f545-459c-adca-034546de6c08, 29, Finished, Available, Finished, False)

root
 |-- adjuster_id: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- license_number: string (nullable = true)
 |-- license_state: string (nullable = true)
 |-- specializations: string (nullable = true)
 |-- hire_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- home_office_id: string (nullable = true)
 |-- supervisor_email: string (nullable = true)
 |-- max_active_claims: integer (nullable = true)



---
## 🚀 Step 4 — Deploy Semantic Model

Uses `create_insurance_sm.py` (already in `./builtin/`) and the pre-downloaded
`InsuranceSM.SemanticModel` files to:

1. Install **fabric-cicd** (if needed)
2. Patch `expressions.tmdl` with this workspace's OneLake URL
3. Publish the model via `fabric-cicd`
4. Trigger a full refresh and wait for completion

> Run the single cell below.

In [28]:
import importlib.util
from pathlib import Path

# Load create_insurance_sm.py as a module from builtin/
spec = importlib.util.spec_from_file_location(
    "create_insurance_sm", "./builtin/create_insurance_sm.py"
)
sm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sm)

sm.ensure_fabric_cicd()

# Patch the pre-downloaded expressions.tmdl then publish
sm_dir = Path("./builtin/InsuranceSM.SemanticModel")
sm.patch_expressions(sm_dir, WORKSPACE_ID, LAKEHOUSE_ID)
sm.publish_model(WORKSPACE_ID, Path("./builtin"))

sm.trigger_refresh(WORKSPACE_ID)

StatementMeta(, 1cc01447-f545-459c-adca-034546de6c08, 30, Finished, Available, Finished, False)

  fabric-cicd 0.3.1 already installed.
  Semantic model deployed successfully.
  Found semantic model: InsuranceSM (4421c235-820a-4333-b71b-ca818577d28d)
  Refresh triggered. Polling every 15 seconds...

    [01] status=Completed     endTime=2026-03-23T20:18:47.32Z

  Semantic model refreshed successfully.


[info]   20:18:18 - Executing as User 'admin@MngEnvMCAP471467.onmicrosoft.com'
[info]   20:18:18 - Relative directory path 'builtin' resolved as '/synfs/resource/nb_resource/builtin'
[info]   20:18:18 - 
[info]   20:18:18 - ####################################################################################################
[info]   20:18:18 - ########## Validating Parameter File ###############################################################
[info]   20:18:18 - ####################################################################################################
[info]   20:18:18 - 
[warn]   20:18:19 - Parameter file not found with path: /synfs/resource/nb_resource/builtin/parameter.yml
[warn]   20:18:19 - Validation terminated: not found
[info]   20:18:20 - 
[info]   20:18:20 - ####################################################################################################
[info]   20:18:20 - ########## Publishing Workspace Folders ###################################################

---
## 🏠 Step 5 — Create Eventhouse & KQL Database

Uses `create_insurance_eventhouse.py` (already in `./builtin/`) to create:
- **Eventhouse** `eh_insurance`
- **KQL Database** `insurance_db` inside it

Both resources are idempotent — existing ones are detected and skipped.

> Provisioning is asynchronous; the cell polls until both are ready.

In [29]:
import importlib.util

# Load create_insurance_eventhouse.py as a module from builtin/
spec = importlib.util.spec_from_file_location(
    "create_insurance_eventhouse", "./builtin/create_insurance_eventhouse.py"
)
eh = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eh)

client = eh._get_client()

print("[1/3] Eventhouse...")
EVENTHOUSE_ID = eh.ensure_eventhouse(WORKSPACE_ID, client)

print("\n[2/3] KQL Database...")
KQL_DB_ID = eh.ensure_kql_database(WORKSPACE_ID, EVENTHOUSE_ID, client)

print(f"\n✅ Eventhouse  : {eh.EVENTHOUSE_NAME} ({EVENTHOUSE_ID})")
print(f"   KQL Database : {eh.KQL_DB_NAME} ({KQL_DB_ID})")

print("\n[3/3] KQL Tables...")
created = eh.create_kql_tables(WORKSPACE_ID, EVENTHOUSE_ID, eh.KQL_DB_NAME, client)
print(f"\n✅ Tables ready: {', '.join(created) if created else '(none)'}")


StatementMeta(, 1cc01447-f545-459c-adca-034546de6c08, 31, Finished, Available, Finished, False)

[1/3] Eventhouse...
  Creating Eventhouse 'eh_insurance'...
  Provisioning eh_insurance... done.
  ✅ Eventhouse created — EVENTHOUSE_ID = 1344092d-4cd4-46fb-98ae-fcf7ea0d5d69

[2/3] KQL Database...
  Creating KQL Database 'insurance_db' inside Eventhouse...
  Provisioning insurance_db.... done.
  ✅ KQL Database created — KQL_DB_ID = a70bdff1-6826-4255-b538-d2b15e4efbf2

✅ Eventhouse  : eh_insurance (1344092d-4cd4-46fb-98ae-fcf7ea0d5d69)
   KQL Database : insurance_db (a70bdff1-6826-4255-b538-d2b15e4efbf2)

[3/3] KQL Tables...
  ✅ [1/6] .create-merge table ClaimStatusEvent (
  ✅ [2/6] .alter table ClaimStatusEvent policy retention ```{ "SoftDeletePeriod": "365.00:
  ✅ [3/6] .create-merge table FraudAlertEvent (
  ✅ [4/6] .create-merge table InspectionEvent (
  ✅ [5/6] .create-merge table PolicyChangeEvent (
  ✅ [6/6] .create-merge table PaymentEvent (

✅ Tables ready: ClaimStatusEvent, FraudAlertEvent, InspectionEvent, PolicyChangeEvent, PaymentEvent


In [8]:
print("KUSTO_URI =", KUSTO_URI)
print("EVENTHOUSE_ID =", EVENTHOUSE_ID)
print("KUSTO_DATABASE =", KUSTO_DATABASE if "KUSTO_DATABASE" in locals() else "not set")

StatementMeta(, 07cefa52-e009-41d4-ae35-3b50dc96b221, 10, Finished, Available, Finished, False)

NameError: name 'KUSTO_URI' is not defined

### Verify KQL Tables

In [30]:
import requests

# Resolve Eventhouse query URI
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/eventhouses/{EVENTHOUSE_ID}")
resp.raise_for_status()
KUSTO_URI = resp.json()["properties"]["queryServiceUri"]
KUSTO_DATABASE = eh.KQL_DB_NAME

kusto_token = notebookutils.credentials.getToken("https://kusto.kusto.windows.net")
headers = {
    "Authorization": f"Bearer {kusto_token}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}
mgmt_url = f"{KUSTO_URI}/v1/rest/mgmt"

# Verify tables exist in the KQL database
verify_stmt = ".show tables | project TableName"
body = {"db": KUSTO_DATABASE, "csl": verify_stmt, "properties": {}}
r = requests.post(mgmt_url, headers=headers, json=body, timeout=60)
r.raise_for_status()

rows = r.json().get("Tables", [{}])[0].get("Rows", [])
existing = sorted(row[0] for row in rows)

expected = {"ClaimStatusEvent", "FraudAlertEvent", "InspectionEvent", "PolicyChangeEvent", "PaymentEvent"}
print(f"Tables in '{KUSTO_DATABASE}':\n")
for name in existing:
    status = "✅" if name in expected else "  "
    print(f"  {status} {name}")

missing = expected - set(existing)
if missing:
    print(f"\n⚠️  Missing tables: {', '.join(sorted(missing))}")
else:
    print(f"\n✅ All {len(expected)} event tables confirmed.")

StatementMeta(, 1cc01447-f545-459c-adca-034546de6c08, 32, Finished, Available, Finished, False)

Tables in 'insurance_db':

  ✅ ClaimStatusEvent
  ✅ FraudAlertEvent
  ✅ InspectionEvent
  ✅ PaymentEvent
  ✅ PolicyChangeEvent

✅ All 5 event tables confirmed.


---
## ⚡ Step 6 — Generate Synthetic Events

Runs `02_generate_events.ipynb` as a child notebook, passing the Eventhouse
query URI and KQL Database name as parameters.

The notebook generates realistic claim-status, fraud-alert, inspection,
policy-change, and payment event streams and ingests them directly into the
`insurance_db` KQL tables via the Kusto Python SDK.

In [15]:
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/eventhouses/{EVENTHOUSE_ID}")
resp.raise_for_status()
props = resp.json()["properties"]

KUSTO_URI = props["queryServiceUri"]
INGEST_URI = props["ingestionServiceUri"]
KUSTO_DATABASE = eh.KQL_DB_NAME

print(f"  Eventhouse Query URI : {KUSTO_URI}")
print(f"  Eventhouse Ingest URI: {INGEST_URI}")
print(f"  KQL Database         : {KUSTO_DATABASE}")

notebookutils.notebook.run(
    "02_generate_events",
    timeout_seconds=900,
    arguments={
        "KUSTO_URI": KUSTO_URI,
        "INGEST_URI": INGEST_URI,
        "KUSTO_DATABASE": KUSTO_DATABASE,
        "LAKEHOUSE_ID": LAKEHOUSE_ID,
        "WORKSPACE_ID": WORKSPACE_ID,
    },
)

print("\n✅ Event generation complete.")

StatementMeta(, 07cefa52-e009-41d4-ae35-3b50dc96b221, 17, Finished, Available, Finished, False)

  Eventhouse Query URI : https://trd-dskkr6cdrtznbj3w9w.z9.kusto.fabric.microsoft.com
  Eventhouse Ingest URI: https://ingest-trd-dskkr6cdrtznbj3w9w.z9.kusto.fabric.microsoft.com
  KQL Database         : insurance_db



✅ Event generation complete.


## Step 7: Create Ontology

Creates the Fabric Ontology if `create_ontology = True`. Set to `False` to create manually — see README Option 2.

In [42]:
if create_ontology:
    print("Running 03_create_ontology notebook...")
    notebookutils.notebook.run(
        "03_create_ontology",
        timeout_seconds=900,
        arguments={
            "WORKSPACE_NAME": WORKSPACE_NAME,
            "LAKEHOUSE_NAME": LAKEHOUSE_NAME,
        }
    )
    print("✅ Ontology created successfully.")
else:
    print("ℹ️  create_ontology = False — skipping automated creation.")
    print("Follow README Option 2 to create the ontology manually from the semantic model.")


StatementMeta(, 1cc01447-f545-459c-adca-034546de6c08, 44, Finished, Available, Finished, False)

Running 03_create_ontology notebook...


✅ Ontology created successfully.
